# ECG image classification: Google Colab inference example

This example runs the research models to distinguish Takotsubo syndrome (`tako`) from acute anterior ST-segment elevation myocardial infarction (STEMI, labeled `AMI` in the output). It calls the same code as the local CLI.

The default test-time augmentation (TTA) settings are **ViT: TTA 24 / magnitude 1, ConvNeXtV2: no TTA, and MLP-Mixer: TTA 32 / magnitude 3**. The ensemble takes an **equally weighted mean of the models' raw logits, then applies softmax**. Each TTA count includes the original, unaugmented view. You can change the settings in the cells below.

Trained weights are distributed as a separate GitHub Release asset. By default, this notebook downloads the public inference weights and verifies their SHA-256 checksums. Patient images are not included; provide an image that you are authorized to use in Colab. Patient images do not need to be made public. Checking agreement with the paper's existing predictions requires the corresponding private images and prediction results, which are used separately.

## 1. Select the code version

The default `GIT_REF = "v0.2.0"` selects the planned release containing the inference code and public weight downloader. **The tag and weight asset will be available after the v0.2.0 release is published.** The actual checked-out commit SHA is recorded alongside the results.

This notebook requires **Python 3.11 or later** and a code revision containing the security fixes, live progress support, and release weight downloader. `v0.1.1` does not contain the CLI. Older inference revisions do not contain all of these features.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import re
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Colab runtime with Python 3.11 or later.")

ENV_DIR = Path("/content/ecg_inference_environment")

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "v0.2.0"  # Available after publication of this release.
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("Set GIT_REF to a branch name, tag name, or commit SHA.")

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError("REPO_DIR is not an existing Git working directory. Choose another location.")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("The repository in REPO_DIR does not match REPO_URL.")
    if git("status", "--porcelain"):
        raise RuntimeError("The working directory has changes. Save them before running this cell again.")

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = git("rev-parse", "HEAD")
required_code = [
    REPO_DIR / "examples" / "predict_ensemble.py",
    REPO_DIR / "requirements-inference.txt",
    REPO_DIR / "src" / "takotsubo_ami_ai_ecg" / "security.py",
    REPO_DIR / "src" / "takotsubo_ami_ai_ecg" / "progress.py",
    REPO_DIR / "tools" / "download_weights.py",
    REPO_DIR / "src" / "takotsubo_ami_ai_ecg" / "assets" / "weights_v0.2.0.json",
]
if any(not path.is_file() for path in required_code):
    raise RuntimeError(
        "The selected revision lacks required inference or weight-download files. Set GIT_REF to v0.2.0 or a revision containing all required files."
    )
print("Code commit:", CODE_COMMIT)
sys.path.insert(0, str(REPO_DIR / "src"))


## 2. Install inference dependencies

To use a GPU, select one in Colab's runtime settings. CPU execution is also supported, but running all three models with TTA can take time.

The installation preserves Colab's existing PyTorch/torchvision pair and adds the inference dependencies. It uses the patched Pillow version 13.0.0. The root `requirements.txt`, which records the historical training environment, is not used in this cell.

**If installation changes any package versions, select Runtime > Restart session, then run the code cells again in order, starting with the first code cell.** The restart error intentionally stops execution at this point. Repeating the same cell without restarting will not allow execution to continue. Restarting clears Python variables, so do not resume from a cell in the middle of the notebook.


In [ ]:
ENVIRONMENT_READY = False
ENV_DIR.mkdir(parents=True, exist_ok=True)
restart_marker = ENV_DIR / "restart_required.json"


def check_session_restart():
    if restart_marker.is_file():
        recorded = json.loads(restart_marker.read_text(encoding="utf-8"))
        if recorded["pid"] == os.getpid():
            raise RuntimeError(
                "Package versions have changed. Select Runtime > Restart session, "
                "then run the code cells in order, starting with the first code cell."
            )
        restart_marker.unlink()


def installed_versions():
    return {
        dist.metadata["Name"].lower().replace("_", "-"): dist.version
        for dist in metadata.distributions()
        if dist.metadata.get("Name")
    }


try:
    torch_version = metadata.version("torch")
    torchvision_version = metadata.version("torchvision")
except metadata.PackageNotFoundError as exc:
    raise RuntimeError("Select a Colab runtime with PyTorch and torchvision installed.") from exc

constraints_path = ENV_DIR / "colab_torch_constraints.txt"
constraints_path.write_text(
    f"torch=={torch_version}\ntorchvision=={torchvision_version}\n", encoding="utf-8"
)
before_install = installed_versions()
installation = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-inference.txt"),
     "-c", str(constraints_path)],
    check=False,
)
after_install = installed_versions()
changed_packages = sorted(
    name for name in before_install.keys() | after_install.keys()
    if before_install.get(name) != after_install.get(name)
)
if changed_packages:
    restart_marker.write_text(
        json.dumps({"pid": os.getpid(), "changed_packages": changed_packages}), encoding="utf-8"
    )
    print("Changed packages:", ", ".join(changed_packages))
if installation.returncode != 0:
    raise RuntimeError(
        "Dependency installation failed. Review the pip error above."
        + (" Package versions changed, so restart the session before trying again." if changed_packages else "")
    )
check_session_restart()
ENVIRONMENT_READY = True
print("Preserved PyTorch / torchvision:", torch_version, "/", torchvision_version)


In [ ]:
if not globals().get("ENVIRONMENT_READY", False):
    raise RuntimeError("Run the setup cells in order, from the first code cell through dependency installation.")
check_session_restart()

from takotsubo_ami_ai_ecg.security import require_patched_pillow
require_patched_pillow()

import torch
import torchvision
from PIL import Image

runtime_versions = {"python": sys.version.split()[0]}
for package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors"):
    runtime_versions[package] = metadata.version(package)
runtime_versions["cuda_available"] = torch.cuda.is_available()
runtime_versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(json.dumps(runtime_versions, ensure_ascii=False, indent=2))


## 3. Obtain the inference weights

By default, the weight-preparation cell downloads the public **v0.2.0** Release asset and verifies the archive and extracted files against the SHA-256 checksums bundled with this code revision. The asset becomes available after the release is published. Download status is displayed as it runs. Weights are stored at `/content/ecg_inference_weights`, outside the repository checkout, so rerunning the notebook does not create changes in the Git working directory. Complete, verified files are reused on subsequent runs.

The planned weight terms are **CC BY-NC 4.0 for ViT** (noncommercial use, with upstream Apache notices) and **MIT for ConvNeXtV2 and MLP-Mixer**. The code remains MIT-licensed. Read [WEIGHTS_LICENSE.md](../WEIGHTS_LICENSE.md) before downloading.

For weights already in Colab or Google Drive, set `WEIGHT_SOURCE = "local"` and change `WEIGHTS_DIR` to their directory. Mount Drive in the following cell only if an image or your local weights are on Drive. The public download does not require Drive.

Both modes use the following layout relative to `WEIGHTS_DIR`:

- `vit/model.safetensors` and `vit/config.json`
- `convnextv2/model.safetensors`
- `mlpmixer/model.safetensors`

For local weights, use files created by `tools/export_inference_weights.py`; do not upload ordinary training checkpoints directly. You may also retain ViT's `preprocessor_config.json` as a record of its settings. To match the original dataset evaluation code, inference normalizes inputs using the ImageNet mean and standard deviation for all three models. ConvNeXtV2 uses the regular model weights from `ckpt_0011.pth`, not the SWA weights.


In [ ]:
USE_GOOGLE_DRIVE = False  # Set to True only when reading files from Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
WEIGHTS_READY = False
check_session_restart()

WEIGHT_SOURCE = "release"  # "release" for verified public weights; "local" for existing files.
WEIGHTS_DIR = Path("/content/ecg_inference_weights")
# Examples for WEIGHT_SOURCE = "local":
# WEIGHTS_DIR = Path("/content/weights")
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")

if WEIGHT_SOURCE not in {"release", "local"}:
    raise ValueError('Set WEIGHT_SOURCE to "release" or "local".')
if WEIGHT_SOURCE == "release":
    download_command = [
        sys.executable, "-u", str(REPO_DIR / "tools" / "download_weights.py"),
        "--output-dir", str(WEIGHTS_DIR),
    ]
    download_process = None
    try:
        download_process = subprocess.Popen(
            download_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in download_process.stdout:
            print(line, end="", flush=True)
        download_returncode = download_process.wait()
        if download_returncode != 0:
            raise RuntimeError(
                f"Weight download or verification failed with exit code {download_returncode}. "
                "Review the error above. The v0.2.0 asset is available only after the release is published."
            )
    except BaseException:
        if download_process is not None and download_process.poll() is None:
            download_process.terminate()
            try:
                download_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                download_process.kill()
                download_process.wait(timeout=5)
        raise
    finally:
        if download_process is not None and download_process.stdout is not None:
            download_process.stdout.close()
else:
    if not WEIGHTS_DIR.is_dir():
        raise FileNotFoundError("Set WEIGHTS_DIR to the existing inference-weight directory.")
    print("Using local inference weights; no download requested.")

WEIGHTS_READY = True
print("Weights directory:", WEIGHTS_DIR)


## 4. Configure the input image and inference settings

Upload an authorized image using Colab's Files panel, or select an image from Drive if you mounted it above. Set `IMAGE_PATH` below. Patient images are not downloaded with the public weights. Results are saved to `/content/ecg_inference_results` by default.


In [ ]:
# Choose one of: "ensemble", "vit", "convnextv2", "mlpmixer".
MODEL = "ensemble"

# Set the path to an input image that you are authorized to use in Colab.
IMAGE_PATH = Path("/content/data/example.png")
# Example when using Drive:
# IMAGE_PATH = Path("/content/drive/MyDrive/ecg_inference/data/example.png")

OUTPUT_DIR = Path("/content/ecg_inference_results")
# Output filenames are generated automatically each time the inference cell runs.

# Paper TTA settings. TTA=1 uses only the untransformed image.
VIT_TTA = 24
VIT_MAGNITUDE = 1
CONVNEXTV2_TTA = 1
CONVNEXTV2_MAGNITUDE = 5  # Unused when TTA=1.
MLPMIXER_TTA = 32
MLPMIXER_MAGNITUDE = 3
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]  # Order: ViT, ConvNeXtV2, MLP-Mixer.

THRESHOLD = None  # None: AMI probability >= 0.5 for the ensemble; argmax for a single model. Set a number to override.
SEED = 1234
DEVICE = "auto"  # "auto", "cpu", "cuda"
PRECISION = "float32"  # "float32" or "bfloat16"
INPUT_POLICY = "strict480"  # Set explicitly to "resize" to resize inputs to 480x480.
BATCH_SIZE = 1
NUM_WORKERS = 4  # Original evaluation setting. Set to 0 if needed.

## 5. Check the inputs and run inference

`strict480` requires **480 x 480 pixel** inputs, matching the original dataset evaluation. If your images have a different size, check that the preprocessing matches training. `resize` is an explicit alternative and does not guarantee that the image preparation matches the paper.

The seed controls TTA randomness. However, results may not exactly match the original batch evaluation if image order, batch size, worker count, libraries, or numerical precision differ.
Each run of the inference cell creates a new output filename containing a timestamp and an identifier. Results with no TTA and with the paper's TTA settings can be saved as separate JSON files and run records.

The inference cell displays a live progress bar with overall completion, the current model, TTA view, image count, and elapsed time. The percentage counts completed image/view evaluations; it is not a time estimate and may pause while models load or results are saved. Logs and errors appear as they arrive. Stopping the cell also stops its inference subprocess. The final success message appears after both the prediction and matching run record have been saved.


In [ ]:
check_session_restart()
require_patched_pillow()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before checking inputs.")

supported_models = ("ensemble", "vit", "convnextv2", "mlpmixer")
if MODEL not in supported_models:
    raise ValueError(f"Choose MODEL from {supported_models}.")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("Provide the input image and set IMAGE_PATH to its location.")

selected_models = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
required_weights = [WEIGHTS_DIR / model / "model.safetensors" for model in selected_models]
if "vit" in selected_models:
    required_weights.append(WEIGHTS_DIR / "vit" / "config.json")
if any(not path.is_file() for path in required_weights):
    raise FileNotFoundError("Inference weights or configuration files are missing for the selected model(s). Check the layout above.")

with Image.open(IMAGE_PATH) as image:
    input_size = image.size
if INPUT_POLICY == "strict480" and input_size != (480, 480):
    raise ValueError(f"The input dimensions are {input_size} pixels. strict480 requires a 480x480 image.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input dimensions:", input_size)
print("Selected model:", MODEL)

In [ ]:
result = None

from datetime import datetime, timezone
import hashlib
from html import escape
import math
import time
import uuid
from IPython.display import HTML, display

check_session_restart()
require_patched_pillow()
if not globals().get("WEIGHTS_READY", False):
    raise RuntimeError("Complete the weight-preparation cell successfully before running inference.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_JSON = OUTPUT_DIR / f"prediction_{MODEL}_{stamp}_{uuid.uuid4().hex[:8]}.json"
RUN_JSON = OUTPUT_JSON.with_suffix(".colab_run.json")

command = [
    sys.executable, "-u",
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--progress", "json",
]
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])

progress_prefix = "ECG_PROGRESS "
progress_state = {"completed": 0, "total": 0, "message": "Starting inference."}
progress_display = None
process = None
progress_error = False
started = time.monotonic()


def finite_number(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default


def count_pair(current, total):
    if current is None or total is None:
        return "n/a"
    return f"{escape(str(current))}/{escape(str(total))}"


def render_progress(message=None):
    global progress_display
    total = max(0.0, finite_number(progress_state.get("total")))
    completed = max(0.0, finite_number(progress_state.get("completed")))
    percent = min(100.0, 100.0 * completed / total) if total else 0.0
    elapsed = max(time.monotonic() - started, finite_number(progress_state.get("elapsed_seconds")))
    model = progress_state.get("model") or "n/a"
    model_position = count_pair(progress_state.get("model_index"), progress_state.get("model_count"))
    tta = count_pair(progress_state.get("tta_index"), progress_state.get("tta_total"))
    images = count_pair(progress_state.get("images_completed"), progress_state.get("images_total"))
    status = escape(str(message if message is not None else progress_state.get("message", "Running inference.")))
    content = HTML(
        '<div style="padding:10px 0;max-width:760px">'
        f'<div><strong>{status}</strong></div>'
        f'<progress value="{percent:.2f}" max="100" style="width:100%;height:20px" '
        'aria-label="Inference progress"></progress>'
        f'<div>{percent:.1f}% | Model: {escape(str(model))} ({model_position}) '
        f'| TTA: {tta} | Images: {images} | Elapsed: {elapsed:.1f} s</div></div>'
    )
    if progress_display is None:
        progress_display = display(content, display_id=True)
    else:
        progress_display.update(content)


def stop_inference_process():
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)


try:
    render_progress()
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        if line.startswith(progress_prefix):
            try:
                event = json.loads(line[len(progress_prefix):])
            except json.JSONDecodeError:
                print("An unreadable progress update was skipped.", flush=True)
                continue
            if not isinstance(event, dict) or event.get("event") not in {"stage", "progress", "complete", "error"}:
                print("An unrecognized progress update was skipped.", flush=True)
                continue
            progress_state.update(event)
            if event["event"] == "error":
                progress_error = True
                render_progress("Inference failed. Review the logs below.")
            elif event["event"] == "complete":
                render_progress("Prediction saved; finishing the run record.")
            else:
                render_progress()
        else:
            print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Inference stopped with exit code {returncode}. Review the error displayed above."
        )
    if progress_error:
        raise RuntimeError("Inference reported an error. Review the logs above.")

    pending_result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
    # Link the output to its run record without recording the absolute input path.
    run_record = {
        "code_commit": CODE_COMMIT,
        "prediction_file": OUTPUT_JSON.name,
        "prediction_sha256": hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest(),
        "runtime": runtime_versions,
        "inference_versions": pending_result["metadata"]["versions"],
        "actual_device": pending_result["metadata"]["actual_device"],
    }
    with RUN_JSON.open("x", encoding="utf-8") as stream:
        json.dump(run_record, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    result = pending_result
    render_progress("Inference completed; prediction and run record saved.")
except BaseException as exc:
    result = None
    try:
        stop_inference_process()
    finally:
        failure_message = "Inference cancelled." if isinstance(exc, KeyboardInterrupt) else "Inference failed. Review the error below."
        try:
            render_progress(failure_message)
        except Exception:
            print(failure_message, flush=True)
    raise
finally:
    if process is not None and process.stdout is not None:
        process.stdout.close()

print("Inference completed.")
print("Prediction:", OUTPUT_JSON.name)
print("Run record:", RUN_JSON.name)


## 6. Review the results

The class order is **0 = tako, 1 = AMI**. By default, the ensemble uses `prob_AMI >= 0.5`, while individual models use argmax (a probability tie is assigned to AMI for the ensemble and tako for an individual model). If you set a numeric `THRESHOLD`, it is used for either case. The following cell displays model outputs. Without comparison against ground-truth labels, these outputs do not establish sensitivity, specificity, or accuracy.

The JSON records the actual execution settings, weight identifiers, logits, and probabilities. Colab's working storage is temporary, so save any results you need to retain.
Successful inference creates `prediction_<model>_<timestamp>_<id>.json` and a matching file ending in `.colab_run.json`. The latter records the code commit, runtime environment, and SHA-256 of the prediction file. Use the download cell below to save both files.


In [ ]:
if not isinstance(globals().get("result"), dict):
    raise RuntimeError("Complete inference successfully before reviewing the results.")

for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Set to True if needed to download the two JSON files from the same run.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    if not isinstance(globals().get("result"), dict) or not OUTPUT_JSON.is_file() or not RUN_JSON.is_file():
        raise RuntimeError("Confirm that inference completed successfully and both the prediction and run record were saved.")
    from google.colab import files
    files.download(str(OUTPUT_JSON))
    files.download(str(RUN_JSON))


## Validation scope

On 2026-10-22, the inference code before the security update was checked against the legacy evaluation code on ten real ECG images in WSL, using float32 with no TTA. One synthetic image was also processed on a Colab A100 using the paper's TTA settings and float32. Validation of the revision containing the Pillow update and loading restrictions is recorded separately from those checks. See `docs/implementation_status.md` for the current status.

On 2026-10-22, the author reported a successful Colab run with live progress at commit `6f4b59c`. The new Release-download workflow in this notebook has not yet been executed on Colab.

Comparison with the paper's existing predictions requires the same weights, input images, and preprocessing. Numerical agreement with historical predictions made using TTA has not been established by the checks completed so far.

The distributed notebook contains no patient images or saved execution results. Before sharing an executed copy, clear all outputs, including images, predictions, and logs.
